# Combinatorial CoWork 2026 — Notebook 03: Change of posets and common refinement

Audience:
- Participants who want to see functoriality, transport, and common refinement as first-class workflow tasks rather than ad hoc storage manipulations.

Learning goals:
- Use `common_refinement(...)` as the canonical notebook-facing transport path.
- Inspect the translated modules and the projection maps without unpacking fields manually.
- Demonstrate `restriction`, `pushforward_left`, `pushforward_right`, and their derived versions on a tiny explicit finite map.


## Outline

1. Setup and two separately encoded box examples
2. Translate them to a common refinement
3. Use `hom(...; transport=:common_refinement)`
4. Pull one module back along a projection map
5. One clearly labeled advanced aside for explicit pushforwards on a tiny finite map


In [ ]:
    NOTEBOOK_STEM = "03_change_of_posets_and_common_refinement"

_TO_ROOT = let
    dir = abspath(pwd())
    root = nothing
    while true
        if isfile(joinpath(dir, "src", "TamerOp.jl"))
            root = dir
            break
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repo root containing src/TamerOp.jl from pwd()=$(pwd()).")
        dir = parent
    end
    root
end

if !isdefined(Main, :TO) || !isdefined(Main, :example_header)
    Base.include(Main, joinpath(_TO_ROOT, "examples", "_common.jl"))
end

CM = TO.CoreModules
OPT = TO.Options
sc = TO.SessionCache()
outdir = joinpath(_TO_ROOT, "examples", "_outputs", "combinatorial_cowork_2026", NOTEBOOK_STEM)
mkpath(outdir)

nothing


## 1. Two small box presentations that do not share an encoding a priori

We encode them separately on purpose so that common refinement has real work to do.


In [ ]:
SD = TO.SyntheticData
box_left = SD.box_bar_fringe(
    bars=[([0.0, 0.0], [1.25, 1.0]), ([1.0, 0.25], [2.0, 1.5])],
    field=CM.QQField(),
)
box_right = SD.staircase_box_fringe(
    bars=[([0.0, 0.0], [1.5, 1.0]), ([0.75, 0.25], [2.25, 1.75]), ([1.5, 0.75], [3.0, 2.5])],
    field=CM.QQField(),
)
enc_left = TO.encode(box_left; cache=sc)
enc_right = TO.encode(box_right; cache=sc)
left_regions_spec = TO.Visualization.visual_spec(TO.encoding_map(enc_left); kind=:regions)

(; left=TO.describe(enc_left),
   right=TO.describe(enc_right),
   left_regions_summary=TO.Visualization.visual_summary(left_regions_spec))


## 2. Common refinement is the canonical notebook-facing transport path

This is the surface we want participants to remember when two encoded modules do not already share a classifier map.


In [ ]:
cref = TO.common_refinement(enc_left, enc_right; cache=sc)
mods = TO.translated_modules(cref)
pis = TO.projection_maps(cref)

(; common_refinement=TO.describe(cref),
   left_translated=TO.describe(mods.left),
   right_translated=TO.describe(mods.right),
   common_hom_dim=TO.hom_dimension(enc_left, enc_right; transport=:common_refinement, cache=sc),
   common_hom=TO.describe(TO.hom(enc_left, enc_right; transport=:common_refinement, cache=sc)))


## 3. Restriction along a projection map

The projection maps returned by `common_refinement(...)` let us explain pullback/restriction directly. This cell should match the translated-left module in the common-refinement result.


In [ ]:
left_pull = TO.restriction(pis.left, enc_left; cache=sc)
right_pull = TO.restriction(pis.right, enc_right; cache=sc)

(; left_pull=TO.describe(left_pull),
   right_pull=TO.describe(right_pull),
   translated_left_matches_poset=(TO.encoding_poset(left_pull) === TO.common_poset(cref)),
   translated_right_matches_poset=(TO.encoding_poset(right_pull) === TO.common_poset(cref)))


## 4. Advanced aside: one tiny explicit finite map for pushforwards

This is the only deliberate advanced cell in the core suite. We build a two-point chain onto a one-point chain so that the left/right pushforward story is completely explicit.


In [ ]:
TOA = TO.Advanced
Psmall = TOA.FinitePoset(reshape(Bool[1], 1, 1); check=false)
Qbig = TOA.FinitePoset(Bool[1 1; 0 1]; check=false)

pi_q_to_p = TOA.EncodingMap(Qbig, Psmall, [1, 1])
id_P = TOA.EncodingMap(Psmall, Psmall, [1])
id_Q = TOA.EncodingMap(Qbig, Qbig, [1, 2])

field = CM.QQField()
H_small = TOA.one_by_one_fringe(Psmall, TOA.principal_upset(Psmall, 1), TOA.principal_downset(Psmall, 1), TO.QQ(1); field=field)
H_big = TOA.one_by_one_fringe(Qbig, TOA.principal_upset(Qbig, 1), TOA.principal_downset(Qbig, 2), TO.QQ(1); field=field)

enc_small = TO.EncodingResult(Psmall, TOA.pmodule_from_fringe(H_small), id_P; H=H_small, opts=OPT.EncodingOptions(field=field), backend=:workshop)
enc_big = TO.EncodingResult(Qbig, TOA.pmodule_from_fringe(H_big), id_Q; H=H_big, opts=OPT.EncodingOptions(field=field), backend=:workshop)

(; enc_small=TO.describe(enc_small), enc_big=TO.describe(enc_big), pi_map=TO.describe(pi_q_to_p))


## 5. Left/right pushforwards and their derived versions

Now that the map is explicit, the change-of-posets operations read like ordinary functorial constructions.


In [ ]:
left = TO.pushforward_left(pi_q_to_p, enc_big; cache=sc)
right = TO.pushforward_right(pi_q_to_p, enc_big; cache=sc)
df_opts = OPT.DerivedFunctorOptions(maxdeg=1)
dleft = TO.derived_pushforward_left(pi_q_to_p, enc_big; opts=df_opts, cache=sc)
dright = TO.derived_pushforward_right(pi_q_to_p, enc_big; opts=df_opts, cache=sc)

(; left=TO.describe(left),
   right=TO.describe(right),
   derived_left=map(TO.describe, dleft),
   derived_right=map(TO.describe, dright))


## 6. Export the region views

Use the same `save_visuals(...)` surface here so participants see that export is uniform across workflow objects.


In [ ]:
exports = TO.save_visuals(
    outdir,
    [
        (; stem="left_regions", obj=TO.encoding_map(enc_left), kind=:regions),
        (; stem="right_regions", obj=TO.encoding_map(enc_right), kind=:regions),
    ];
    format=:html,
    backend=:cairomakie,
)

Dict(TO.export_stem(r) => TO.export_path(r) for r in exports)


## Try this next

Change one bar in `box_right`, rerun the common-refinement cells, and compare the translated-module summaries and `common_hom_dim`.


In [ ]:
box_right_variant = SD.staircase_box_fringe(
    bars=[([0.0, 0.0], [1.25, 1.0]), ([0.75, 0.25], [2.5, 1.5]), ([1.75, 1.0], [3.25, 2.75])],
    field=CM.QQField(),
)
enc_right_variant = TO.encode(box_right_variant; cache=sc)
cref_variant = TO.common_refinement(enc_left, enc_right_variant; cache=sc)

(; right_variant=TO.describe(enc_right_variant),
   variant_common_refinement=TO.describe(cref_variant),
   variant_common_hom_dim=TO.hom_dimension(enc_left, enc_right_variant; transport=:common_refinement, cache=sc))
